# Mountain car sim environment

In [ ]:
%pip install picogym

In [ ]:
from picogym.mountain_car.widget import MountainCarWidget

env = MountainCarWidget(manual_control=True)
env.render()

In [ ]:
from picogym.mountain_car.widget import MountainCarWidget
import asyncio

env = MountainCarWidget()
env.render()

# This is an example of a solution that fails to reach the goal.
# actions = [2 if t % 50 < 10 else 0 for t in range(200)]

# This is an example of a solution that reaches the goal.
actions = [a for a in [0, 2, 0, 2, 0, 2] for _ in range(60)]

for action in actions:
    state = env.step(action)
    await asyncio.sleep(0.01)

## We also provide a tkinker frontend

In [ ]:
from picogym.mountain_car.tk import MountainCarTkFrontend
import asyncio

env = MountainCarTkFrontend()
env.render()
actions = [a for a in [0, 2, 0, 2, 0, 2] for _ in range(60)]

for action in actions:
    state = env.step(action)
    await asyncio.sleep(0.01)

# Now try to use reinforcement learning to solve the problem!

In [ ]:
import numpy as np
from tqdm.auto import tqdm
from picogym.mountain_car import MountainCarEnv

env = MountainCarEnv()
NUM_BINS = (40, 40)  # (position, velocity)
low_pos, high_pos = env.min_position, env.max_position
low_vel, high_vel = -env.max_speed, env.max_speed
Q = np.zeros((*NUM_BINS, env.action_space.n))


def discretize_state(state):
    pos_bin = int((state["position"] - low_pos) / (high_pos - low_pos) * NUM_BINS[0])
    vel_bin = int((state["velocity"] - low_vel) / (high_vel - low_vel) * NUM_BINS[1])
    pos_bin = np.clip(pos_bin, 0, NUM_BINS[0] - 1)
    vel_bin = np.clip(vel_bin, 0, NUM_BINS[1] - 1)
    return pos_bin, vel_bin


# Hyperparameters
alpha = 0.1
gamma = 0.99
epsilon = 0.1

pbar = tqdm(range(1500), desc="Training Episodes")
completed, min_steps = False, float("inf")

for _ in pbar:
    state = env.reset()
    s = discretize_state(state)

    for step in range(400):
        if np.random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(Q[s])

        # classic reward heuristic (-1 reward per step)
        next_state = env.step(action)
        s2 = discretize_state(next_state)
        Q[s][action] += alpha * (-1 + gamma * np.max(Q[s2]) - Q[s][action])
        s = s2

        if next_state["done"]:
            completed = True
            min_steps = min(min_steps, step)
            break

    pbar.set_postfix(
        {"status": f"goal, min steps: {min_steps}" if completed else "failed"}
    )

In [ ]:
from picogym.mountain_car.widget import MountainCarWidget
import asyncio

env = MountainCarWidget()
env.render()

state = env.reset()
s = discretize_state(state)

for step in range(500):
    await asyncio.sleep(0.01)
    action = np.argmax(Q[s])
    next_state = env.step(action)
    s2 = discretize_state(next_state)
    Q[s][action] += alpha * (-1 + gamma * np.max(Q[s2]) - Q[s][action])
    s = s2

## Vectorize Environments

In [ ]:
from picogym.mountain_car.tk import MountainCarTkFrontend
from picogym.mountain_car import MountainCarEnv
import asyncio
import numpy as np

num_envs = 4
env = MountainCarTkFrontend(sim_env=MountainCarEnv(num_envs=num_envs))
env.render()

base_actions = np.array([0, 2, 0, 2, 0, 2])
time_steps = 60 * len(base_actions)

# Random actions for each environment
actions = np.stack(
    [np.random.permutation(np.tile(base_actions, 60)) for _ in range(num_envs)],
    axis=1,
)

for action in actions:
    env.step(action)
    await asyncio.sleep(0.01)